In [ ]:
#Reads the files in the directory and creates a TileDB array from them
import tiledb
import numpy as np
import os
outdir = "/scratch/bruno.ariano/chr22_txt_parsed/chr22_B_test_compressed/"
tiledb.from_csv("/scratch/bruno.ariano/tiledb_array_test_scattered", 
                [os.path.join(outdir, f) for f in os.listdir(outdir)],
                tile=500000000,
                chunksize=30000000,
                sparse=True, 
                index_dims=['cell_type','gene','position'],
                column_types={"SNP" : str, "position":np.float64, "gene": str, "beta": np.float64, "p-value": np.float64, "cell_type": str},
                engine = 'c'
                )

In [ ]:
lower_pos = 16849573
higher_pos = 18890804
cell_type = ["CD4_T","CD8_T","CD16_Mono","CD14_Mono","B","NK"]

In [ ]:
#Single query filter not optimised on Dask and withouth dumping data
A_scratch = tiledb.open("/scratch/bruno.ariano/test_spark_tiledb_single_cell_chr22", mode="r")
A_scratch.query(return_arrow=True, dims=['gene', 'cell_type'], attrs=['p-value', 'beta']).df[lower_pos:higher_pos, :, cell_type]

In [ ]:
#Batch query filter optimised on Dask and with dumping of the data
import os
from dask import delayed, compute
import pyarrow as pa
import pyarrow.csv
import pyarrow.parquet as pq
import gc
# Define the chunk size based on your data and available resources
chunk_size = 20000
start_positions = range(lower_pos, higher_pos, chunk_size)

@delayed
def batch_query_tiledb(array, queries, output_dir, batch_index):
    results = []
    for start, stop, cell_type in queries:
        result = array.query(return_arrow=True, dims=['gene', 'cell_type'], attrs=['p-value', 'beta', 'SNP']).df[start:stop, :, cell_type]
        results.append(result)
    
    # Concatenate all results into a single PyArrow Table
    combined_result = pa.concat_tables(results)
    
    # Check if the combined result has at least one row
    if combined_result.num_rows > 0:
        # Define the output file path
        output_path = os.path.join(output_dir, f"batch_{batch_index}.csv")
        
        # Write the combined result to a CSV file
        pyarrow.csv.write_csv(combined_result, output_path, write_options=pa.csv.WriteOptions(include_header=True))
        #gc.collect()
        return output_path
    else:
        # Return None if the combined result is empty
        #gc.collect()
        return None

# Create queries and batch them
queries = [(start, min(start + chunk_size, higher_pos), cell) for cell in cell_type for start in start_positions]
batched_queries = [queries[i:i + 5] for i in range(0, len(queries), 5)]

# Define the directory where the CSV files will be saved
output_dir = "/scratch/bruno.ariano/output_gene"
os.makedirs(output_dir, exist_ok=True)

# Run the computation and store the paths of generated CSV files
result_paths = compute(*[batch_query_tiledb(A_scratch, batch, output_dir, idx) for idx, batch in enumerate(batched_queries)])

# Filter out None values from the result_paths list
result_paths = [path for path in result_paths if path is not None]

# Print the paths of the generated CSV files
print("Generated CSV files:", result_paths)

In [ ]:
import pandas as pd
t = 1
for chunk in pd.read_csv('/scratch/nicola.pirastu/Onek1k_eQTLs_chr22_raw_counts_trans_B.txt.gz', compression='gzip', chunksize=5000000, delimiter = "\t", engine = 'c'):
    
    # Step 3: Extract the position from the SNP column in each chunk
    chunk['position'] = chunk['SNP'].str.split(':').str[1].str.split('_').str[0]
    
    # Step 4: Drop the t-stat column in each chunk
    chunk = chunk.drop(columns=['t-stat'])
    output_filename = f'/scratch/bruno.ariano/chr22_txt_parsed/chr22_B'
    chunk.to_csv(output_filename, index=False, compression='gzip')
    t = t+1
    # Convert the Pandas chunk to a Dask DataFrame and add it to the list
    

# Step 5: Combine all Dask DataFrames into one



In [ ]:
#This is a faster and less computationally expensive way to create many small chunk of processed gz files

import pandas as pd
import dask
from dask import delayed

# Step 1: Function to process each chunk
@delayed
def process_and_save_chunk(chunk, chunk_index):
    # Step 2: Extract the position from the SNP column
    chunk['position'] = chunk['SNP'].str.split(':').str[1].str.split('_').str[0]
    
    # Step 3: Drop the t-stat column
    chunk = chunk.drop(columns=['t-stat'])
    
    # Step 4: Write the chunk to a separate CSV file
    output_filename = f'/scratch/bruno.ariano/scratch/bruno.ariano/chr22_txt_parsed/chr22_B_chunk_{chunk_index}.csv.gz'
    chunk.to_csv(output_filename, index=False, compression='gzip')
    
    return output_filename

# Step 5: List to hold the delayed tasks
tasks = []

# Step 6: Read the gzipped CSV file in chunks and create delayed tasks
# Read the gzipped CSV file in chunks
for i, chunk in enumerate(pd.read_csv('/scratch/nicola.pirastu/Onek1k_eQTLs_chr22_raw_counts_trans_B.txt.gz', 
                                      compression='gzip', chunksize=5000000, delimiter="\t")):
    # Process and save each chunk immediately
    delayed_process = delayed(process_and_save_chunk)(chunk, i)
    delayed_process.compute()  # Immediately compute the task and release memory